In [16]:
#Lab 4.3 —  LSTMs & GRUs on Text
#1. Dataset prep
import torch
import torch.nn as nn
import torch.nn.functional as F

text = "hello world this is a character level lstm example"

# Create vocabulary
chars = sorted(set(text))

# Character -> ID
char_to_idx = {ch: i for i, ch in enumerate(chars)}

# Convert text to integer IDs
seq = torch.tensor(
    [char_to_idx[ch] for ch in text],
    dtype=torch.long
)

seq_len = 25

X = []
Y = []

for i in range(len(seq) - seq_len):
    X.append(seq[i:i + seq_len])
    Y.append(seq[i + 1:i + seq_len + 1])

X = torch.stack(X)
Y = torch.stack(Y)

print("Vocabulary size:", len(chars))
print("X shape:", X.shape)
print("Y shape:", Y.shape)


Vocabulary size: 17
X shape: torch.Size([25, 25])
Y shape: torch.Size([25, 25])


In [18]:
#2. Model definitions

class CharLSTM(nn.Module):
    def __init__(self, vocab, hidden=128):
        super().__init__()

        self.emb = nn.Embedding(vocab, 32)
        self.lstm = nn.LSTM(32, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, vocab)

    def forward(self, x):
        o, _ = self.lstm(self.emb(x))
        return self.fc(o)


class CharGRU(nn.Module):
    def __init__(self, vocab, hidden=128):
        super().__init__()

        self.emb = nn.Embedding(vocab, 32)
        self.gru = nn.GRU(32, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, vocab)

    def forward(self, x):
        o, _ = self.gru(self.emb(x))
        return self.fc(o)


# Create model AFTER chars is defined
model = CharLSTM(len(chars))

opt = torch.optim.Adam(
    model.parameters(),
    lr=1e-2
)

In [19]:
#3. Training Loop
for ep in range(5):

    logits = model(X)

    loss = F.cross_entropy(
        logits.reshape(-1, len(chars)),
        Y.reshape(-1)
    )

    opt.zero_grad()
    loss.backward()
    opt.step()

    print("epoch", ep, "loss", loss.item())

epoch 0 loss 2.850987672805786
epoch 1 loss 2.6441617012023926
epoch 2 loss 2.3512635231018066
epoch 3 loss 2.270062208175659
epoch 4 loss 1.9237697124481201


In [23]:
#4. Text Generation
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)} 

def generate(model, start="r", length=50):
    model.eval()

    out = start
    idx = torch.tensor([[stoi[ch] for ch in start]], dtype=torch.long)

    for _ in range(length):
        with torch.no_grad():
            emb = model.emb(idx)
            logits, _ = model.lstm(emb)

            nxt = torch.argmax(
                model.fc(logits[:, -1, :]),
                dim=1
            ).item()

        out += itos[nxt]
        idx = torch.tensor([[nxt]], dtype=torch.long)

    return out


print(generate(model))

rachachachachachachachachachachachachachachachachac
